# Experiment 3 — a join with no equality condition

`orders` (2 M rows) joined to three price bands by a range condition. Spark UI: http://localhost:4040 → **SQL / DataFrame**.

In [5]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m4-exp3-non-equi").getOrCreate()
orders = spark.read.parquet("/opt/data/m1/orders")
bands = spark.createDataFrame([(0.0, 100.0, "low"), (100.0, 300.0, "mid"), (300.0, 600.0, "high")], "lo double, hi double, band string")

cond = (orders.amount >= bands.lo) & (orders.amount < bands.hi)

def timed(label, df):
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

## A — default settings

**PREDICT:** the join node's name; whether anything is shuffled.

In [6]:
a = orders.join(bands, cond)
a.explain("formatted")
timed("A default", a)

== Physical Plan ==
CartesianProduct Inner (6)
:- * Filter (3)
:  +- * ColumnarToRow (2)
:     +- Scan parquet  (1)
+- * Filter (5)
   +- * Scan ExistingRDD (4)


(1) Scan parquet 
Output [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(amount)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]

(3) Filter [codegen id : 1]
Input [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]
Condition : isnotnull(amount#29)

(4) Scan ExistingRDD [codegen id : 2]
Output [3]: [lo#31, hi#32, band#33]
Arguments: [lo#31, hi#32, band#33], MapPartitionsRDD[24] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(5) Filter [codegen id : 2]
Input [3]: [lo#31, hi#3

[Stage 5:==============================================>          (52 + 8) / 64]

A default: 3.2s


## B — automatic broadcasting disabled

**PREDICT:** the join node's name now.

In [7]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
b = orders.join(bands, cond)
b.explain("formatted")
timed("B no auto-broadcast", b)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10MB")

== Physical Plan ==
CartesianProduct Inner (6)
:- * Filter (3)
:  +- * ColumnarToRow (2)
:     +- Scan parquet  (1)
+- * Filter (5)
   +- * Scan ExistingRDD (4)


(1) Scan parquet 
Output [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(amount)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]

(3) Filter [codegen id : 1]
Input [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]
Condition : isnotnull(amount#29)

(4) Scan ExistingRDD [codegen id : 2]
Output [3]: [lo#31, hi#32, band#33]
Arguments: [lo#31, hi#32, band#33], MapPartitionsRDD[24] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(5) Filter [codegen id : 2]
Input [3]: [lo#31, hi#3

[Stage 6:=========================================>               (47 + 8) / 64]

B no auto-broadcast: 1.0s


In [8]:
c = orders.join(F.broadcast(bands), cond)
c.explain("formatted")
timed("C broadcast hint", c)

== Physical Plan ==
AdaptiveSparkPlan (7)
+- BroadcastNestedLoopJoin Inner BuildRight (6)
   :- Filter (2)
   :  +- Scan parquet  (1)
   +- BroadcastExchange (5)
      +- Filter (4)
         +- Scan ExistingRDD (3)


(1) Scan parquet 
Output [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(amount)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date>

(2) Filter
Input [5]: [order_id#26L, customer_id#27, shop_id#28, amount#29, order_date#30]
Condition : isnotnull(amount#29)

(3) Scan ExistingRDD
Output [3]: [lo#31, hi#32, band#33]
Arguments: [lo#31, hi#32, band#33], MapPartitionsRDD[24] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(4) Filter
Input [3]: [lo#31, hi#32, band#33]
Condition : (isnotnull(lo#31) AND isnotnull(hi#32))

(5) BroadcastExchange
Input [3]: [lo#31, hi#32, 

[Stage 8:>                                                          (0 + 8) / 8]

C broadcast hint: 2.3s


## Stop before opening the next notebook

In [ ]:
spark.stop()